In [ ]:
# package import

import psycopg2
import pandas as pd
import geopandas as gpd
from geopandas import GeoSeries, GeoDataFrame

import math
import numpy as np
import os
import time
import gc
import glob as gb

from tqdm import tqdm
from shapely import wkt
from shapely import geometry
from shapely.geometry import Polygon, Point , MultiPoint, LineString, MultiPolygon
from shapely.ops import unary_union

import matplotlib.pyplot as plt

In [ ]:
#setting 
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'

In [ ]:
#read shape file 
jung_shp=gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/중구.shp', encoding='utf-8-sig')
dong_shp=gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/동구.shp', encoding='utf-8-sig')
seo_shp=gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/서구.shp', encoding='utf-8-sig')
youseong_shp=gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/유성구.shp', encoding='utf-8-sig')
daedeok_shp=gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/대덕구.shp', encoding='utf-8-sig')

In [ ]:
jung_shp

In [ ]:
cty_shp = gpd.GeoDataFrame(pd.concat([jung_shp, dong_shp, seo_shp, youseong_shp, daedeok_shp], ignore_index=True))

In [ ]:
cty_shp.info()

In [ ]:
ax=cty_shp.plot()
plt.show()

In [ ]:
cty_shp.crs

In [ ]:
cty_shp.to_crs(epsg=5179)
cty_shp.crs

In [ ]:
dst = 500
#target_shp=cty_shp[]  #시도 하나만 하도록!
#---------------  making bbox
# 몇개 셀로 나눠야할지 계산
x_cnt = int(np.ceil((cty_shp.unary_union.envelope.bounds[2]-cty_shp.unary_union.envelope.bounds[0])/dst)) 
y_cnt = int(np.ceil((cty_shp.unary_union.envelope.bounds[3]-cty_shp.unary_union.envelope.bounds[1])/dst))
print(x_cnt,y_cnt)

In [ ]:
#기준 지점 그리기 
orign = [cty_shp.unary_union.envelope.bounds[0], cty_shp.unary_union.envelope.bounds[1]]

# grid cell 만들기
polygons = []
for xi in range(x_cnt):
    for yj in range(y_cnt):
        x1 = orign[0] + (dst * xi)
        x2 = x1 + dst
        y1 = orign[1] + (dst * yj)
        y2 = y1 + dst
        polygons.append(Polygon([(x1,y1), (x1,y2), (x2,y2), (x2,y1)]))

cty_grid = gpd.GeoDataFrame({'geometry': polygons}, crs=cty_shp.crs)

cty_grid = cty_grid[cty_grid.geometry.intersects(cty_shp.unary_union)]

cty_grid.reset_index(drop=True, inplace=True)

In [ ]:
cty_grid.boundary.plot()
plt.show()

In [ ]:
fig,ax = plt.subplots(1,1,sharex=True,sharey=True,figsize=(11,11))
cty_shp.plot(ax=ax)
cty_grid.boundary.plot(ax=ax,color='red')
plt.show()

In [ ]:
cty_grid

# 차고지

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. CSV 파일을 읽어 DataFrame으로 변환
csv_file_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/차고지/대전광역시 차고지 위치데이터.csv"
data = pd.read_csv(csv_file_path)

# 2. 차고지 데이터를 지리정보 데이터프레임으로 변환
gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data['경도'], data['위도']),
    crs="EPSG:4326"  # WGS84 좌표계
)

# 4. 차고지 데이터의 좌표계를 대전 구별 경계 파일의 좌표계로 변환
gdf = gdf.to_crs(cty_shp.crs)

cty_grid['차고지'] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)].shape[0])

# 5. 대전 구별 경계와 차고지 위치를 지도에 표시
fig, ax = plt.subplots(figsize=(11, 11))

# 대전 구별 경계를 표시
cty_shp.plot(ax=ax, edgecolor='black', alpha=0.5)
cty_grid.boundary.plot(ax=ax, color='red', linewidth=1)

# 변환된 좌표계의 차고지 위치를 지도 위에 표시
gdf.plot(ax=ax, color='blue', markersize=10)

plt.show()

print(cty_grid[['geometry', '차고지']])
print(cty_grid[["차고지"]].sum())

# 시내버스정류장

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. CSV 파일을 읽어 DataFrame으로 변환
csv_file_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/대중교통/대전광역시 버스정류장 위치정보.csv"
data = pd.read_csv(csv_file_path)

# 2. 차고지 데이터를 지리정보 데이터프레임으로 변환
gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data['경도'], data['위도']),
    crs="EPSG:4326"  # WGS84 좌표계
)

# 4. 차고지 데이터의 좌표계를 대전 구별 경계 파일의 좌표계로 변환
gdf = gdf.to_crs(cty_shp.crs)

cty_grid['시내버스정류장'] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)].shape[0])


# 5. 대전 구별 경계와 차고지 위치를 지도에 표시
fig, ax = plt.subplots(figsize=(11, 11))

# 대전 구별 경계를 표시
cty_shp.plot(ax=ax, edgecolor='black', alpha=0.5)
cty_grid.boundary.plot(ax=ax, color='red', linewidth=1)

# 변환된 좌표계의 차고지 위치를 지도 위에 표시
gdf.plot(ax=ax, color='blue', markersize=10)

plt.show()

print(cty_grid[['geometry', '시내버스정류장']])
print(cty_grid[["시내버스정류장"]].sum())

# 기차역

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. CSV 파일을 읽어 DataFrame으로 변환
csv_file_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/대중교통/대전광역시 기차역 데이터 (크롤링).csv"
data = pd.read_csv(csv_file_path)

# 2. 차고지 데이터를 지리정보 데이터프레임으로 변환
gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data['경도'], data['위도']),
    crs="EPSG:4326"  # WGS84 좌표계
)

# 4. 차고지 데이터의 좌표계를 대전 구별 경계 파일의 좌표계로 변환
gdf = gdf.to_crs(cty_shp.crs)

cty_grid['기차역'] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)].shape[0])


# 5. 대전 구별 경계와 차고지 위치를 지도에 표시
fig, ax = plt.subplots(figsize=(11, 11))

# 대전 구별 경계를 표시
cty_shp.plot(ax=ax, edgecolor='black', alpha=0.5)
cty_grid.boundary.plot(ax=ax, color='red', linewidth=1)

# 변환된 좌표계의 차고지 위치를 지도 위에 표시
gdf.plot(ax=ax, color='blue', markersize=10)

plt.show()
print(cty_grid[['geometry', '기차역']])
print(cty_grid[["기차역"]].sum())

# 시외버스터미널

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. CSV 파일을 읽어 DataFrame으로 변환
csv_file_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/대중교통/시외고속버스터미널 (버스타고).csv"
data = pd.read_csv(csv_file_path)

# 2. 차고지 데이터를 지리정보 데이터프레임으로 변환
gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data['경도'], data['위도']),
    crs="EPSG:4326"  # WGS84 좌표계
)

# 좌표계를 대전 구별 경계 파일의 좌표계로 변환
gdf = gdf.to_crs(cty_shp.crs)

cty_grid['시외버스터미널'] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)].shape[0])

# 7. 결과를 시각화
fig, ax = plt.subplots(figsize=(11, 11))

cty_shp.plot(ax=ax, edgecolor='black', alpha=0.5)
cty_grid.boundary.plot(ax=ax, color='red', linewidth=1)

# 변환된 좌표계의 차고지 위치를 지도 위에 표시
gdf.plot(ax=ax, color='blue', markersize=10)

plt.show()

print(cty_grid[['geometry', '시외버스터미널']])
print(cty_grid[["시외버스터미널"]].sum())

# 도시철도(지하철)

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. CSV 파일을 읽어 DataFrame으로 변환
csv_file_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/대중교통/대전광역시 도시철도 위치정보.csv"
data = pd.read_csv(csv_file_path)

# 2. 차고지 데이터를 지리정보 데이터프레임으로 변환
gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data['경도'], data['위도']),
    crs="EPSG:4326"  # WGS84 좌표계
)

# 좌표계를 대전 구별 경계 파일의 좌표계로 변환
gdf = gdf.to_crs(cty_shp.crs)

cty_grid['도시철도'] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)].shape[0])

# 7. 결과를 시각화
fig, ax = plt.subplots(figsize=(11, 11))

cty_shp.plot(ax=ax, edgecolor='black', alpha=0.5)
cty_grid.boundary.plot(ax=ax, color='red', linewidth=1)

# 변환된 좌표계의 차고지 위치를 지도 위에 표시
gdf.plot(ax=ax, color='blue', markersize=10)

plt.show()

print(cty_grid[['geometry', '도시철도']])
print(cty_grid[["도시철도"]].sum())

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. CSV 파일을 읽어 DataFrame으로 변환
csv_file_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/대중교통/대전광역시 도시철도 시간대별 이동 패턴분석.csv"
data = pd.read_csv(csv_file_path)

# 2. 도시철도 데이터를 지리정보 데이터프레임으로 변환
gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data['경도'], data['위도']),
    crs="EPSG:4326"  # WGS84 좌표계
)

# 3. 좌표계를 대전 구별 경계 파일의 좌표계로 변환
gdf = gdf.to_crs(cty_shp.crs)

# 4. 시간대별로 이용량을 계산하여 격자에 추가
time_slots = [
    "03~04시", "04~05시", "05~06시", "06~07시", "07~08시", "08~09시", "09~10시", 
    "10~11시", "11~12시", "12~13시", "13~14시", "14~15시", "15~16시", "16~17시", 
    "17~18시", "18~19시", "19~20시", "20~21시", "21~22시", "22~23시", "23~00시", 
    "00~01시", "01~02시", "02~03시"
]

# 격자 데이터에 시간대별 이용량을 합산하는 컬럼 추가
for slot in time_slots:
    cty_grid[slot] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)][slot].sum())

# 7. 결과를 시각화
fig, ax = plt.subplots(figsize=(11, 11))

cty_shp.plot(ax=ax, edgecolor='black', alpha=0.5)
cty_grid.boundary.plot(ax=ax, color='red', linewidth=1)

# 변환된 좌표계의 차고지 위치를 지도 위에 표시
gdf.plot(ax=ax, color='blue', markersize=10)

plt.show()

# 8. 결과 확인
print(cty_grid[['geometry'] + time_slots])
print(cty_grid[time_slots].sum())


# 관광지

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. CSV 파일을 읽어 DataFrame으로 변환
csv_file_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/관광지/대전광역시 관광지 데이터.csv"
data = pd.read_csv(csv_file_path, encoding = "euc-kr")

# 2. 차고지 데이터를 지리정보 데이터프레임으로 변환
gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data['경도'], data['위도']),
    crs="EPSG:4326"  # WGS84 좌표계
)

# 좌표계를 대전 구별 경계 파일의 좌표계로 변환
gdf = gdf.to_crs(cty_shp.crs)

cty_grid['관광지'] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)].shape[0])

# 7. 결과를 시각화
fig, ax = plt.subplots(figsize=(11, 11))

cty_shp.plot(ax=ax, edgecolor='black', alpha=0.5)
cty_grid.boundary.plot(ax=ax, color='red', linewidth=1)

# 변환된 좌표계의 차고지 위치를 지도 위에 표시
gdf.plot(ax=ax, color='blue', markersize=10)

plt.show()

print(cty_grid[['geometry', '관광지']])
print(cty_grid[["관광지"]].sum())



# 숙소

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. CSV 파일을 읽어 DataFrame으로 변환
csv_file_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/상권/대전광역시 숙소 정보 데이터_결과.csv"
data = pd.read_csv(csv_file_path, encoding = "utf-8-sig")

# 2. 차고지 데이터를 지리정보 데이터프레임으로 변환
gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data['경도'], data['위도']),
    crs="EPSG:4326"  # WGS84 좌표계
)

# 좌표계를 대전 구별 경계 파일의 좌표계로 변환
gdf = gdf.to_crs(cty_shp.crs)

cty_grid['상호명'] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)].shape[0])
cty_grid['숙소별 관광지 거리 평균'] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)]['관광지와 숙소의 거리'].sum())


# 7. 결과를 시각화
fig, ax = plt.subplots(figsize=(11, 11))

cty_shp.plot(ax=ax, edgecolor='black', alpha=0.5)
cty_grid.boundary.plot(ax=ax, color='red', linewidth=1)

# 변환된 좌표계의 차고지 위치를 지도 위에 표시
gdf.plot(ax=ax, color='blue', markersize=10)

plt.show()

print(cty_grid[['geometry', '상호명', '숙소별 관광지 거리 평균']])
print(cty_grid[["상호명"]].sum())

print(cty_grid[['숙소별 관광지 거리 평균']].sum())

# 공영주차장

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. CSV 파일을 읽어 DataFrame으로 변환
csv_file_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/주차장/대전광역시 유무료 주차장 정보 데이터.csv"
data = pd.read_csv(csv_file_path)

# 2. 차고지 데이터를 지리정보 데이터프레임으로 변환
gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data['경도'], data['위도']),
    crs="EPSG:4326"  # WGS84 좌표계
)

# 3. 좌표계를 대전 구별 경계 파일의 좌표계로 변환
gdf = gdf.to_crs(cty_shp.crs)

# 4. 좌표 정보를 (위도, 경도) 형태로 묶어서 문자열로 변환하는 함수 정의
def extract_coordinates(geometry, gdf):
    points_within = gdf[gdf.geometry.within(geometry)]
    coordinates = [(lat, lon) for lat, lon in zip(points_within['위도'], points_within['경도'])]
    return ', '.join([f"({lat}, {lon})" for lat, lon in coordinates])

# 5. 주차장명을 리스트로 저장하는 함수 정의
def extract_parking_names(geometry, gdf):
    points_within = gdf[gdf.geometry.within(geometry)]
    parking_names = points_within['주차장명'].tolist()
    return ', '.join(parking_names)

# 6. 주차구획수를 리스트로 저장하는 함수 정의
def extract_parking_space_counts(geometry, gdf):
    points_within = gdf[gdf.geometry.within(geometry)]
    parking_space_counts = points_within['주차구획수'].tolist()
    return ', '.join(map(str, parking_space_counts))

# 7. 격자 안의 좌표를 저장하는 '좌표' 컬럼 추가
cty_grid['좌표'] = cty_grid.geometry.apply(lambda x: extract_coordinates(x, gdf))

# 8. 격자 안의 주차장명을 저장하는 '주차장명' 컬럼 추가
cty_grid['주차장명'] = cty_grid.geometry.apply(lambda x: extract_parking_names(x, gdf))

# 9. 격자 안의 주차구획수를 저장하는 '주차구획수' 컬럼 추가
cty_grid['주차구획수'] = cty_grid.geometry.apply(lambda x: extract_parking_space_counts(x, gdf))

# 10. 공영주차장 개수의 합산
cty_grid['공영주차장'] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)].shape[0])

# 11. 결과를 시각화
fig, ax = plt.subplots(figsize=(11, 11))

cty_shp.plot(ax=ax, edgecolor='black', alpha=0.5)
cty_grid.boundary.plot(ax=ax, color='red', linewidth=1)

# 변환된 좌표계의 차고지 위치를 지도 위에 표시
gdf.plot(ax=ax, color='blue', markersize=10)

plt.show()

# 12. 결과 출력
print(cty_grid[['geometry', '공영주차장', '주차구획수', '좌표', '주차장명']])

# 13. 공영주차장 합계 출력
print(cty_grid[["공영주차장"]].sum())


# 상권매출

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. CSV 파일을 읽어 DataFrame으로 변환
csv_file_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/상권/대전광역시 상권매출 수정.csv"
data = pd.read_csv(csv_file_path)

# 2. 차고지 데이터를 지리정보 데이터프레임으로 변환
gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(data['경도'], data['위도']),
    crs="EPSG:4326"  # WGS84 좌표계
)

# 좌표계를 대전 구별 경계 파일의 좌표계로 변환
gdf = gdf.to_crs(cty_shp.crs)

# 공영주차장 개수와 주차구획수의 합산
cty_grid['상권'] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)].shape[0])
cty_grid['상권매출액'] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)]['매출액_int'].sum())
# 5. 상권매출액 평균 계산
cty_grid['상권매출액_평균'] = cty_grid['상권매출액'] / cty_grid['상권'].replace(0, 1)  # 0으로 나누는 것을 방지하기 위해 1로 대체

# 7. 결과를 시각화
fig, ax = plt.subplots(figsize=(11, 11))

cty_shp.plot(ax=ax, edgecolor='black', alpha=0.5)
cty_grid.boundary.plot(ax=ax, color='red', linewidth=1)

# 변환된 좌표계의 차고지 위치를 지도 위에 표시
gdf.plot(ax=ax, color='blue', markersize=10)

plt.show()

print(cty_grid[['geometry', '상권매출액']])
print(cty_grid[["상권매출액"]].sum())

In [ ]:
cty_grid

In [ ]:
# CSV 파일 저장 경로
csv_file_path = "C:/Users/kfq/Desktop/cty_grid.csv"

# 데이터프레임을 CSV 파일로 저장 (utf-8-sig 인코딩)
cty_grid.to_csv(csv_file_path, encoding='utf-8-sig')

# ------------------------------------------------------------------------------------------------

In [ ]:
# GeoJSON 파일 경로
geojson_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/대전광역시 행정동.geojson"

# GeoJSON 파일 읽기
gdf.to_crs(epsg=5179)
gdf = gpd.read_file(geojson_path)

print(gdf)

# Optional: GeoDataFrame 시각화
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(ax=ax, edgecolor='black', alpha=0.5)
plt.show()

In [ ]:
# 좌표계 확인 및 변환
if cty_grid.crs != gdf.crs:
    gdf = gdf.to_crs(cty_grid.crs)

# cty_grid의 폴리곤 안에 gdf의 폴리곤이 포함되는지 확인하고 열 추가
def assign_adm_sgg(row, gdf):
    # gdf에서 현재 cty_grid의 폴리곤을 포함하는 폴리곤을 찾음
    matching_gdf = gdf[gdf.geometry.intersects(row.geometry)]
    if not matching_gdf.empty:
        # intersecting하는 gdf의 첫 번째 폴리곤을 사용
        adm_nm = matching_gdf.iloc[0]['adm_nm']
        sggnm = matching_gdf.iloc[0]['sggnm']
        return pd.Series([adm_nm, sggnm], index=['행정동', '시군구'])
    else:
        return pd.Series([None, None], index=['행정동', '시군구'])

# 새 열 추가
cty_grid[['행정동', '시군구']] = cty_grid.apply(assign_adm_sgg, gdf=gdf, axis=1)

# '행정동' 컬럼의 값을 공백으로 구분하여 세 번째 인덱스를 추출
cty_grid['행정동'] = cty_grid['행정동'].str.split().str[2]  # 세 번째 인덱스(0-based)


In [ ]:

# '행정동' 컬럼에서 null 값을 가진 폴리곤만 필터링
null_cty_grid = cty_grid[cty_grid['행정동'].isna()]

# 시각화
fig, ax = plt.subplots(figsize=(11, 11))

# 대전광역시 행정동 경계 표시
cty_shp.plot(ax=ax, edgecolor='black', alpha=0.5)

# null 값을 가진 cty_grid 폴리곤의 경계 표시
null_cty_grid.boundary.plot(ax=ax, color='red', linewidth=1)

# 제목 추가 (선택 사항)
ax.set_title('Polygons with Null Values in "행정동"', fontsize=15)

plt.show()

In [ ]:
# '행정동' 컬럼에서 null 값을 가진 행 제거
cty_grid_cleaned = cty_grid.dropna(subset=['행정동'])

# 인덱스 초기화
cty_grid_cleaned = cty_grid_cleaned.reset_index(drop=True)

In [ ]:
# 인구현황 CSV 파일 불러오기
population_data_path = "C:/Users/kfq/Downloads/대전광역시 구별 동별 연령별 인구현황.csv"
population_data = pd.read_csv(population_data_path)

# 인구현황 데이터를 '행정동' 컬럼 기준으로 cty_grid_cleaned에 병합
cty_grid_merged = cty_grid_cleaned.merge(population_data[['행정동', '10대 미만', '10대', '20대', '30대', '40대', '50대', '60대', '70대', '80대', '90대', '100대']],
                                         how='left',
                                         left_on='행정동',
                                         right_on='행정동')

# 병합 후 결과 확인
print(cty_grid_merged.head())

In [ ]:
# 유동인구 CSV 파일 불러오기
floating_population_data_path = "C:/Users/kfq/Downloads/대전광역시 구별 유동인구 수(2020.12).csv"
floating_population_data = pd.read_csv(floating_population_data_path)

# 유동인구 데이터의 필요 컬럼만 선택하여 병합
columns_to_merge = ['시군구', '20대', '30대']
cty_grid_merged = cty_grid_merged.merge(floating_population_data[columns_to_merge],
                                        how='left',
                                        left_on='시군구',
                                        right_on='시군구')

# 병합된 데이터에서 20대, 30대 인구수 컬럼명을 변경
cty_grid_merged.rename(columns={'20대': '20대 인구수', '30대': '30대 인구수'}, inplace=True)

# 병합 후 결과 확인
print(cty_grid_merged.head())

In [ ]:
# CSV 파일 저장 경로
csv_file_path = "C:/Users/kfq/Desktop/cty_grid_merged_final2.csv"

# 데이터프레임을 CSV 파일로 저장 (utf-8-sig 인코딩)
cty_grid_merged.to_csv(csv_file_path, encoding='utf-8-sig')

In [ ]:
# # 1. 격자 중심 좌표 계산
# def calculate_centroid(row):
#     polygon = row['geometry']
#     return polygon.centroid

# # 중심 좌표 추가
# cty_grid_merged['centroid'] = cty_grid_merged.apply(calculate_centroid, axis=1)
# cty_grid_merged['centroid_x'] = cty_grid_merged['centroid'].x
# cty_grid_merged['centroid_y'] = cty_grid_merged['centroid'].y

# # 결과 확인
# print(cty_grid_merged[['centroid', 'centroid_x', 'centroid_y']].head())

In [ ]:
# import networkx as nx
# import numpy as np

# # 2. 네트워크 그래프 생성
# def create_network_graph(df, radius=1):
#     G = nx.Graph()
    
#     # 노드 추가 (격자 인덱스와 중심 좌표를 저장)
#     for idx, row in df.iterrows():
#         G.add_node(idx, value=row['차고지'], pos=(row['centroid_x'], row['centroid_y']))
    
#     # 엣지 추가 (인접 격자)
#     for idx1, row1 in df.iterrows():
#         pos1 = (row1['centroid_x'], row1['centroid_y'])
#         for idx2, row2 in df.iterrows():
#             if idx1 != idx2:
#                 pos2 = (row2['centroid_x'], row2['centroid_y'])
#                 distance = np.linalg.norm(np.array(pos1) - np.array(pos2))  # 거리 계산
#                 if distance <= radius:
#                     G.add_edge(idx1, idx2, weight=distance)
    
#     return G

# # 네트워크 그래프 생성
# G = create_network_graph(cty_grid_merged, radius=1000)  # radius 값은 거리 단위에 따라 조정 필요

# # 결과 확인
# print("2. 네트워크 그래프 생성 결과")
# print("총 노드 수:", len(G.nodes))
# print("총 엣지 수:", len(G.edges))
# print("그래프의 일부 노드 및 엣지:")
# print("노드:", list(G.nodes)[:5])
# print("엣지:", list(G.edges)[:5])


In [ ]:
# # 3. 격자별 점수화
# def calculate_score(node, graph):
#     own_value = graph.nodes[node]['value']
#     adjacent_values = [graph.nodes[n]['value'] for n in graph.neighbors(node)]
#     average_adjacent_value = np.mean(adjacent_values) if adjacent_values else 0
#     score = own_value + 0.5 * average_adjacent_value
#     return score

# # 점수 계산
# cty_grid_merged['차고지_점수'] = [calculate_score(node, G) for node in G.nodes]

# # 결과 확인
# print("3. 격자별 점수화 결과")
# print(cty_grid_merged[['차고지', '차고지_점수']].head())

# # 점수화된 차고지 총합과 원래 차고지 총합 확인
# print("점수화된 차고지 총합:", cty_grid_merged['차고지_점수'].sum())
# print("원래 차고지 총합:", cty_grid_merged['차고지'].sum())


In [ ]:
# import networkx as nx
# import numpy as np

# # 1. 네트워크 그래프 생성 함수
# def create_network_graph(df, radius=1):
#     G = nx.Graph()
    
#     for idx, row in df.iterrows():
#         G.add_node(idx, pos=(row['centroid_x'], row['centroid_y']))
#         for column in columns_to_score:
#             G.nodes[idx][column] = row[column]
    
#     # 엣지 추가 (인접 격자)
#     for idx1, row1 in df.iterrows():
#         pos1 = (row1['centroid_x'], row1['centroid_y'])
#         for idx2, row2 in df.iterrows():
#             if idx1 != idx2:
#                 pos2 = (row2['centroid_x'], row2['centroid_y'])
#                 distance = np.linalg.norm(np.array(pos1) - np.array(pos2))  # 거리 계산
#                 if distance <= radius:
#                     G.add_edge(idx1, idx2, weight=distance)
    
#     return G

# # 2. 점수 계산 함수
# def calculate_score(node, graph, column):
#     own_value = graph.nodes[node][column]
#     adjacent_values = [graph.nodes[n][column] for n in graph.neighbors(node)]
#     average_adjacent_value = np.mean(adjacent_values) if adjacent_values else 0
#     score = own_value + 0.5 * average_adjacent_value
#     return score

# # 점수를 계산할 컬럼 리스트
# columns_to_score = ['차고지', '시내버스정류장', '기차역', '시외버스터미널', '도시철도', '관광지', '공영주차장']

# # 3. 네트워크 그래프 생성
# G = create_network_graph(cty_grid_merged, radius=1000)  # radius 값은 거리 단위에 따라 조정 필요

# # 4. 각 컬럼에 대해 점수 계산 및 새로운 컬럼 추가
# for column in columns_to_score:
#     score_column_name = f"{column}_점수"
#     cty_grid_merged[score_column_name] = [calculate_score(node, G, column) for node in G.nodes]

# # 결과 확인
# print("각 컬럼별 점수화 결과")
# print(cty_grid_merged[[col for col in cty_grid_merged.columns if '_점수' in col]].head())

# # 원래 값과 점수화된 값의 총합 비교
# for column in columns_to_score:
#     score_column_name = f"{column}_점수"
#     print(f"{column} 총합: {cty_grid_merged[column].sum()}, 점수화된 {score_column_name} 총합: {cty_grid_merged[score_column_name].sum()}")


In [ ]:
# cty_grid_merged

In [ ]:
# # 대전시 전체 총인구.shp 파일 읽기
# pop_shapefile = gpd.read_file(r'C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\대전 데이터\지도\인구\총인구\대전시 전체\대전시 전체 총인구.shp')

In [ ]:
# pop_shapefile

In [ ]:
# cty_grid_merged

In [ ]:
# from pyproj import CRS
# import geopandas as gpd

# # 좌표계 확인
# print(pop_shapefile.crs)
# print(cty_grid_merged.crs)

# # cty_grid_merged의 좌표계를 EPSG:5179로 설정
# cty_grid_merged = cty_grid_merged.to_crs(CRS.from_epsg(5179))

# # 좌표계 변환 (만약 다르다면)
# if pop_shapefile.crs != cty_grid_merged.crs:
#     pop_shapefile = pop_shapefile.to_crs(cty_grid_merged.crs)


In [ ]:
# # 빈 총인구_격자 컬럼 생성
# cty_grid_merged['총인구_격자'] = 0

# # 공간 조인 수행
# for idx, row in cty_grid_merged.iterrows():
#     grid_geom = row['geometry']
#     overlapping_pop = pop_shapefile[pop_shapefile['geometry'].intersects(grid_geom)]
#     total_population = overlapping_pop['val'].sum()
#     cty_grid_merged.at[idx, '총인구_격자'] = total_population


In [ ]:
# cty_grid_merged

In [ ]:
# # 유동인구 CSV 파일 불러오기
# df_traffic_path = "C:/Users/kfq/Downloads/대전광역시 추정교통량 데이터(2021).csv"
# df_traffic = pd.read_csv(df_traffic_path)

# # 유동인구 데이터의 필요 컬럼만 선택하여 병합
# columns_to_merge = ['행정동', '전체추정교통량(대/일)', '승용차추정교통량', '버스추정교통량', '화물차추정교통량']
# cty_grid_merged = cty_grid_merged.merge(df_traffic[columns_to_merge],
#                                         how='left',
#                                         left_on='행정동',
#                                         right_on='행정동')
# # 병합 후 결과 확인
# print(cty_grid_merged.head())

In [ ]:
# # 대전시 전체 총인구.shp 파일 읽기
# pop_shapefile20 = gpd.read_file(r'C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\대전 데이터\지도\인구\20대\대전시 전체\대전시 전체 20대.shp')

# from pyproj import CRS
# import geopandas as gpd

# # 좌표계 확인
# print(pop_shapefile20.crs)
# print(cty_grid_merged.crs)

# # cty_grid_merged의 좌표계를 EPSG:5179로 설정
# cty_grid_merged = cty_grid_merged.to_crs(CRS.from_epsg(5179))

# # 좌표계 변환 (만약 다르다면)
# if pop_shapefile20.crs != cty_grid_merged.crs:
#     pop_shapefile20 = pop_shapefile20.to_crs(cty_grid_merged.crs)

# # 빈 총인구_격자 컬럼 생성
# cty_grid_merged['20대인구_격자'] = 0

# # 공간 조인 수행
# for idx, row in cty_grid_merged.iterrows():
#     grid_geom = row['geometry']
#     overlapping_pop = pop_shapefile20[pop_shapefile20['geometry'].intersects(grid_geom)]
#     total_population = overlapping_pop['val'].sum()
#     cty_grid_merged.at[idx, '20대인구_격자'] = total_population


In [ ]:
# # 대전시 전체 총인구.shp 파일 읽기
# pop_shapefile30 = gpd.read_file(r'C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\대전 데이터\지도\인구\30대\대전시 전체\대전시 전체 30대.shp')

# from pyproj import CRS
# import geopandas as gpd

# # 좌표계 확인
# print(pop_shapefile30.crs)
# print(cty_grid_merged.crs)

# # cty_grid_merged의 좌표계를 EPSG:5179로 설정
# cty_grid_merged = cty_grid_merged.to_crs(CRS.from_epsg(5179))

# # 좌표계 변환 (만약 다르다면)
# if pop_shapefile30.crs != cty_grid_merged.crs:
#     pop_shapefile30 = pop_shapefile30.to_crs(cty_grid_merged.crs)

# # 빈 총인구_격자 컬럼 생성
# cty_grid_merged['30대인구_격자'] = 0

# # 공간 조인 수행
# for idx, row in cty_grid_merged.iterrows():
#     grid_geom = row['geometry']
#     overlapping_pop = pop_shapefile30[pop_shapefile30['geometry'].intersects(grid_geom)]
#     total_population = overlapping_pop['val'].sum()
#     cty_grid_merged.at[idx, '30대인구_격자'] = total_population


In [ ]:
# # CSV 파일 저장 경로
# csv_file_path = "C:/Users/kfq/Desktop/cty_grid_merged_final.csv"

# # 데이터프레임을 CSV 파일로 저장 (utf-8-sig 인코딩)
# cty_grid_merged.to_csv(csv_file_path, encoding='utf-8-sig')